# BWE Stufe 2 — GAN-Training (Kaggle/GPU)

**Auf Kaggle ausführen.** Setup identisch zu `kaggle_train_regression.ipynb`:

1. **Accelerator → GPU** (T4 ×2 oder P100); braucht ggf. Telefon-Verifizierung.
2. **Internet → On** (sonst scheitert `git clone`).
3. **Add Data → MUSDB18-HQ** einbinden und in Zelle 2 `BWE_DATA_ROOT` darauf zeigen lassen
   (Ordner mit `train/` und `test/`; ein evtl. `val/` wird ignoriert).
4. **Add Data → Phase-C-Run** (Output-Dataset mit `reg_full/generator.weights.h5`) für den
   **Warm-Start** — oder zuerst `kaggle_train_regression.ipynb` in dieser Session laufen lassen.

Das GAN feintunt den **warm-gestarteten** Generator adversarial gegen einen PatchGAN-
Diskriminator (Spectral Norm). `g_loss` ist **kein** Qualitätsmaß — Modellwahl per `val_lsd_hf`
+ Reinhören (Ergebnis-Notebook `03_gan.ipynb`).

In [ ]:
# GPU prüfen
import tensorflow as tf
print("TF", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## 1. Code holen & installieren

TF ist auf Kaggle vorinstalliert → `--no-deps`, damit die GPU-Build nicht ersetzt wird; nur die Audio-Pakete nachinstallieren.

In [ ]:
REPO = "https://github.com/DanyelRychter/bwe_dnn.git"
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO = REPO.replace("https://", f"https://{tok}@")
except Exception:
    pass  # öffentliches Repo

import subprocess
subprocess.run(["git", "clone", "--depth", "1", REPO, "/kaggle/working/bwe_dnn"], check=True)
%pip install -q -e /kaggle/working/bwe_dnn --no-deps
%pip install -q librosa soundfile soxr

## 2. Pfade setzen (VOR dem bwe-Import!)

`BWE_DATA_ROOT` = Ordner mit `train/` und `test/`. Ein evtl. `val/` wird ignoriert (kanonischer Split via Track-Namen).

In [ ]:
import os
# >>> An die tatsächliche Dataset-Struktur anpassen: <<<
os.environ["BWE_DATA_ROOT"] = "/kaggle/input/datasets/quanglvitlm/musdb18-hq"     # muss train/ und test/ enthalten
os.environ["BWE_CKPT_ROOT"] = "/kaggle/working/bwe_runs"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import sys; sys.path.insert(0, "/kaggle/working/bwe_dnn")
from bwe import config as cfg
print(cfg.summary())
print(f"\nGAN: λ_adv={cfg.LAMBDA_ADV}  λ_fm={cfg.LAMBDA_FM}  lr={cfg.GAN_LR}  beta_1={cfg.GAN_BETA_1}  "
      f"D-Vorlauf={cfg.GAN_D_WARMUP_STEPS} Steps")

In [ ]:
# Datensatz-Check: erwartete 86/14/50 (bzw. tatsächliche Zahlen)
from bwe.data import splits as SP
for s in SP.SPLIT_NAMES:
    print(f"{s:6s}: {len(SP.get_split(s)):3d} Tracks")

## 2b. 32-kHz-Cache (optional, wie in `kaggle_train_regression.ipynb` Abschnitt 2b)

Resampelt die 4 Stems aller Tracks **einmal** nach 32 kHz und biegt `cfg.DATA_ROOT` darauf um
→ der On-the-fly-Resample-Zweig entfällt, das Training wird GPU-Bound statt I/O-Bound. Identisch
zum Regressions-Notebook; hier kompakt. Vor dem Training ausführen (resumebar).

In [ ]:
# import os, time
# from pathlib import Path
# import soundfile as sf, soxr
# from bwe.data import splits as SP
# CACHE_ROOT = Path("/kaggle/working/musdb18hq_32k")
# tracks = SP.get_split("train") + SP.get_split("valid") + SP.get_split("test")
# t0 = time.time()
# for i, track in enumerate(tracks, 1):
#     out_dir = CACHE_ROOT / track.subset / track.name; out_dir.mkdir(parents=True, exist_ok=True)
#     for stem in cfg.STEMS:
#         dst = out_dir / f"{stem}.wav"
#         if dst.exists():
#             continue
#         info = sf.info(str(track.stems[stem]))
#         data, sr = sf.read(str(track.stems[stem]), always_2d=True, dtype="float32")
#         if sr != cfg.SR:
#             data = soxr.resample(data, sr, cfg.SR, quality="HQ")
#         sf.write(str(dst), data, cfg.SR, subtype=info.subtype or "PCM_16")
#     if i % 10 == 0 or i == len(tracks):
#         print(f"  [{i:3d}/{len(tracks)}] {time.time()-t0:6.0f}s  {track.name[:42]}")
# os.environ["BWE_DATA_ROOT"] = str(CACHE_ROOT); cfg.DATA_ROOT = CACHE_ROOT
# print("cfg.DATA_ROOT ->", cfg.DATA_ROOT)

## 3. Warm-Start-Gewichte (Phase C)

Sucht `generator.weights.h5` zuerst in eingebundenen Input-Datasets, dann unter `BWE_CKPT_ROOT`
(falls die Regression in derselben Session lief). Ohne Warm-Start würde das GAN from-scratch
starten — möglich, aber instabiler und nicht die geplante Curriculum-Logik.

In [ ]:
import glob
cands = (sorted(glob.glob("/kaggle/input/**/reg*/generator.weights.h5", recursive=True))
         + sorted(glob.glob("/kaggle/input/**/generator.weights.h5", recursive=True))
         + sorted(glob.glob(os.path.join(os.environ["BWE_CKPT_ROOT"], "**", "generator.weights.h5"), recursive=True)))
WARM_START = cands[0] if cands else None
print("Warm-Start:", WARM_START or "KEINER gefunden — GAN startet from-scratch (nicht empfohlen).")

## 4. Subset-GAN (λ-Werte / Lernraten festklopfen)

Wenige Tracks → zuerst prüfen, dass das Training **nicht divergiert** (d_loss/g_loss pendeln,
kein Kollaps) und das HF sichtbar schärfer wird als bei der Regression. Bei Instabilität:
`λ_adv` senken / `λ_fm` erhöhen (in `cfg` oder als `train(...)`-fernes Argument über die CLI).

In [ ]:
from bwe.train.gan import train
model, hist = train(run="gan_subset", mode="subset", warm_start=WARM_START,
                    batch_size=16, epochs=30, limit=20)

## 5. Voll-GAN

Voller Datensatz, resumebar (eigener `tf.train.Checkpoint` über G, D und **beide** Optimizer).
Bricht eine Session ab (12-h-Grenze), Zelle einfach erneut ausführen — der D-Vorlauf wird beim
Resume übersprungen, das Training setzt bei der gesicherten Epoche fort.

> **Kaggle-Hinweis:** `/kaggle/working` ist flüchtig → für Cross-Session-Resume `bwe_runs` als
> Output-Dataset sichern und beim nächsten Start als Input einbinden (dann `BWE_CKPT_ROOT` darauf).

In [ ]:
from bwe.train.gan import train
model, hist = train(run="gan_full", mode="full", warm_start=WARM_START, epochs=cfg.EPOCHS)

## 6. Lernkurven

Gewichte unter `bwe_runs/gan_full/` (`generator.weights.h5` = letzter Stand,
`best_generator.weights.h5` = bestes Val-LSD-HF, `discriminator.weights.h5`). **Wichtig:**
`g_loss`/`d_loss` sind kein Qualitätsmaß (D ist bewegliches Ziel) — sie sollen nur *pendeln*,
nicht kollabieren. Qualität steht in `val_lsd_hf`.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv(os.path.join(os.environ["BWE_CKPT_ROOT"], "gan_full", "log.csv"))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(log["epoch"], log["d_loss"], label="d_loss")
ax[0].plot(log["epoch"], log["g_loss"], label="g_loss")
ax[0].plot(log["epoch"], log["recon"], label="recon", ls="--")
ax[0].set_title("GAN-Losses (kein Qualitätsmaß!)"); ax[0].set_xlabel("Epoche"); ax[0].legend()
ax[1].plot(log["epoch"], log["val_lsd_hf"])
ax[1].set_title("Val-LSD-HF [dB] (Qualitätsmaß)"); ax[1].set_xlabel("Epoche")
plt.tight_layout(); plt.show()